# Task HumanEval

In [ ]:
import os
import re
import sys
import json
import tempfile
import subprocess
from datasets import load_dataset
from datasets import get_dataset_config_names
from datasets import get_dataset_split_names

In [2]:
def print_colored(convo, limit=float('inf')):
    for i, message in enumerate(convo['messages']):
        if i >= limit:
            print(f"\033[31m... {len(convo['messages']) - limit} more messages ...\033[0m")
            break
        role = message['role']
        content = message['content']
        if role == 'system':
            print(f"\033[33m{content}\033[0m")  # yellow
        elif role == 'assistant':
            print(f"\033[34m{content}\033[0m")  # blue
        elif role == 'user':
            print(f"\033[32m{content}\033[0m")  # green
        else:
            print(f"\033[31m{content}\033[0m")  # red

In [3]:
subsets = get_dataset_config_names("openai/openai_humaneval")
print(f"Subsets: {subsets}")


splits = get_dataset_split_names("openai/openai_humaneval")
print(f"Splits: {splits}")

Subsets: ['openai_humaneval']
Splits: ['test']


In [4]:
ds = load_dataset("openai/openai_humaneval", split="test")

In [5]:
example = ds[0]
print(f"Example: {json.dumps(example, indent=2)}")

Example: {
  "task_id": "HumanEval/0",
  "prompt": "from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    \"\"\" Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    \"\"\"\n",
  "canonical_solution": "    for idx, elem in enumerate(numbers):\n        for idx2, elem2 in enumerate(numbers):\n            if idx != idx2:\n                distance = abs(elem - elem2)\n                if distance < threshold:\n                    return True\n\n    return False\n",
  "test": "\n\nMETADATA = {\n    'author': 'jt',\n    'dataset': 'test'\n}\n\n\ndef check(candidate):\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False\n    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95)

In [6]:
for k, v in example.items():
    print(f"--- --- --- {k} --- --- ---")
    print(f"{v}")

--- --- --- task_id --- --- ---
HumanEval/0
--- --- --- prompt --- --- ---
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """

--- --- --- canonical_solution --- --- ---
    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return False

--- --- --- test --- --- ---


METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False
    assert candidate([1.0, 2.0, 5.

In [7]:
prompt = example['prompt']
print(prompt)

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """



In [ ]:
def extract_imports(prompt):
    imports = []
    for line in prompt.splitlines():
        line = line.strip()
        if line.startswith("import ") or line.startswith("from "):
            imports.append(line)
        elif line and not line.startswith("#"):
            break  # break on first non-empty, non-comment line
    return '\n'.join(imports)

In [19]:
imports = extract_imports(prompt)
print(imports)

from typing import List


In [15]:
canonical_solution = example['canonical_solution']
print(canonical_solution)

    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return False



In [23]:
canonical_full_solution = prompt + "\n" + canonical_solution
print(canonical_full_solution)

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """

    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return False



In [30]:
entry_point = example['entry_point']
print(entry_point)

has_close_elements


In [31]:
test = example['test']
print(test)



METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.8) == False
    assert candidate([1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1) == True
    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 1.0) == True
    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 0.5) == False




In [32]:
messages = [
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": canonical_solution},
]
convo = {
    "messages": messages,
    "eval": {
        "imports": imports,
        "entry_point": entry_point,
        "test": test
    }
}

In [33]:
print_colored(convo)

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """

    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return False



In [48]:
CODE_BLOCK_RE = re.compile(r"```(?:python)?\s*\n(.*?)\n```", re.DOTALL)  # Adopted from Nanochat
def extract_program(completion):
    """Extracts program from first ```python or ``` block, returns completion.strip() otherwise."""
    match = CODE_BLOCK_RE.search(completion)
    if match:
        return match.group(1).strip()
    return completion.strip()

example_program_with_test = """\
Here is a simple Python function that adds two numbers:
```python
def add(a, b):
    return a + b

def check(candidate):
    assert candidate(2, 3) == 5
    assert candidate(-1, 1) == 0

check(add)
```
Is there anything else you would like to know about this function?
"""

program = extract_program(example_program_with_test)
print(program)

def add(a, b):
    return a + b

def check(candidate):
    assert candidate(2, 3) == 5
    assert candidate(-1, 1) == 0

check(add)


In [50]:
def execute_code(code, timeout=5, max_memory=512*1024*1024):
    """Run code in a subprocess with temp cwd and resource limits. NOT a secure sandbox! Use with caution."""
    # Explainer - basically prevent obvious breakouts, trivial to bypass by adversarial code
    # - resource.RLIMIT_AS - total process memory limit (heap, stack, mappings, libs, etc.)
    # - builtins.exit/quit - prevent clever model to use quit() to exit with code 0 (success) before test asserts
    # - OMP_NUM_THREADS=1 - prevent excessive threads by native libraries
    # - os.kill - don't kill other processes
    # - os.system - don't run shell commands
    # - os.fork/os.forkpty - don't fork new processes
    # - os.killpg - don't kill/signal process groups
    # - subprocess.Popen - don't spawn new processes
    guard = f"""
import builtins, os, subprocess, resource
limit = {max_memory}
resource.setrlimit(resource.RLIMIT_AS, (limit, limit))
builtins.exit = None
builtins.quit = None
os.environ["OMP_NUM_THREADS"] = "1"
for name in ('kill', 'system', 'fork', 'forkpty', 'killpg'):
    setattr(os, name, None)
subprocess.Popen = None
"""
    # {code!r} - properly escapes code strings, preventing injection attacks (like SQL injection name = "Robert'); DROP TABLE Students;--")
    # compile() - compiles as new program, allowing 'from __future__ ...' to work (needs to be at the top of the file, which guard prevents)
    # exec() - executes compiled code above
    program = guard + f"\nexec(compile({code!r}, '<llm>', 'exec'), {{'__name__': '__main__'}})\n"
    with tempfile.TemporaryDirectory() as tmpdir:
        try:
            process = subprocess.run(
                [sys.executable, "-I", "-c", program],  # -I for isolated mode
                cwd=tmpdir,
                env={},
                stdin=subprocess.DEVNULL,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.PIPE,
                text=True,
                timeout=timeout,
            )
        except subprocess.TimeoutExpired:
            return False, "timeout"
    if process.returncode == 0:
        return True, None
    if "MemoryError" in process.stderr:
        return False, "memory limit exceeded"
    else:
        error_lines = process.stderr.strip().splitlines()
        error = error_lines[-1] if error_lines else "unknown error"
        return False, error

In [51]:
success, error = execute_code(program)
print(f"Execution success: {success}, error: {error}")

Execution success: True, error: None


In [70]:
class TaskHumanEval:
    def __init__(self, split, stop=None):
        assert split == "test"  # HumanEval has only test split
        self.dataset = load_dataset("openai/openai_humaneval", split=split)
        self.length = stop if stop is not None else len(self.dataset)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError(idx)
        example = self.dataset[idx]
        prompt = example['prompt']
        canonical_solution = example['canonical_solution']
        full_solution = f"{prompt}\n{canonical_solution}"
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": full_solution},
        ]
        # according to Nanochat, assistant solutions sometimes omit imports,
        # so we save them and inject during eval as a bit of "help"
        imports = extract_imports(prompt)
        test = example['test']
        entry_point = example['entry_point']
        convo = {
            "messages": messages,
            "eval": {
                "imports": imports,
                "entry_point": entry_point,
                "test": test
            }
        }
        return convo

    def evaluate(self, assistant_response, eval_data, return_error=False):
        assert isinstance(assistant_response, str)
        imports = eval_data["imports"]  # apparently, assistant not always includes imports, so we help a bit
        assistant_program = extract_program(assistant_response)
        test = eval_data["test"]
        entry_point = eval_data["entry_point"]
        full_program = (
            imports + "\n\n"
            + assistant_program + "\n\n"
            + test + "\n\n"
            + f"check({entry_point})"
        )
        success, error = execute_code(full_program)
        if return_error:
            return success, error
        return success

In [53]:
def check_schema(convo):
    assert isinstance(convo, dict)
    assert convo.keys() == {'messages', 'eval'}
    assert isinstance(convo['messages'], list)
    for message in convo['messages']:
        assert isinstance(message, dict)
        assert message.keys() == {'role', 'content'}
        assert message['role'] in {'system', 'assistant', 'user'}
        assert isinstance(message['content'], str)
        assert len(message['content']) > 0
    assert isinstance(convo['eval'], dict)
    assert convo['eval'].keys() == {'imports', 'entry_point', 'test'}

In [71]:
# Checked ok
task = TaskHumanEval("test")
for i, e in enumerate(task):
    check_schema(e)
    if i % 10_000 == 0:
        print(f"Checked {i} / {len(task)} examples...")



Checked 0 / 164 examples...


In [74]:
task = TaskHumanEval("test")
for i, e in enumerate(task):
    assistant_response = e['messages'][-1]['content']
    result, error = task.evaluate(assistant_response, e['eval'], return_error=True)
    if error:
        print(result, error)
    assert result
    result, error = task.evaluate("X", e['eval'], return_error=True)
    assert not result